# CI com GitHub Actions — Bella Tavola 🍝
## Parte 2: Testes automatizados com pytest

# BLOCO 3

## Exercício 3.1 — Preparando a estrutura de testes

In [1]:
# O pipeline passou com o teste mínimo?
# Sim. O step de testes apareceu e ficou verde.

# O step de testes apareceu nos logs da aba Actions?
# Sim. Ele substituiu a etapa anterior de curl e passou a validar o projeto com pytest.

## Exercício 3.2 — Conectando o pipeline ao resultado dos testes

In [ ]:
# Ao adicionar temporariamente um teste que falhava de propósito, o pipeline ficou vermelho.
# A mensagem exibiu AssertionError, mostrando que o GitHub Actions interpreta falhas
# do pytest como falhas do job.

## Exercício 3.3 — Primeiro teste real da API

In [ ]:
# Arquivo: tests/test_pratos.py

from fastapi.testclient import TestClient
from main import app

client = TestClient(app)


def test_raiz_retorna_nome_restaurante():
    response = client.get("/")
    assert response.status_code == 200
    body = response.json()
    assert "restaurante" in body
    assert body["restaurante"] == "Bella Tavola"


def test_listar_pratos_retorna_200():
    response = client.get("/pratos")
    assert response.status_code == 200


def test_listar_pratos_retorna_lista():
    response = client.get("/pratos")
    assert isinstance(response.json(), list)


def test_filtro_categoria_pizza_retorna_apenas_pizzas():
    response = client.get("/pratos?categoria=pizza")
    assert response.status_code == 200
    pratos = response.json()
    for prato in pratos:
        assert prato["categoria"] == "pizza"


def test_buscar_prato_existente_retorna_campos_esperados():
    response = client.get("/pratos/1")
    assert response.status_code == 200
    prato = response.json()
    assert "id" in prato
    assert "nome" in prato
    assert "preco" in prato


def test_buscar_prato_inexistente_retorna_404():
    response = client.get("/pratos/9999")
    assert response.status_code == 404

## Exercício 3.4 — Testando criação e validação

In [ ]:
# Continuação do arquivo: tests/test_pratos.py

def test_criar_prato_valido():
    novo_prato = {
        "nome": "Funghi Trifolati Teste",
        "categoria": "massa",
        "preco": 49.9,
        "descricao": "Prato criado durante o teste",
        "disponivel": True
    }
    response = client.post("/pratos", json=novo_prato)
    assert response.status_code in [200, 201]
    body = response.json()
    assert body["nome"] == novo_prato["nome"]
    assert body["categoria"] == novo_prato["categoria"]
    assert body["preco"] == novo_prato["preco"]
    assert "id" in body


def test_post_prato_preco_negativo_retorna_422():
    prato_invalido = {
        "nome": "Prato Inválido",
        "categoria": "pizza",
        "preco": -10.0
    }
    response = client.post("/pratos", json=prato_invalido)
    assert response.status_code == 422


def test_post_prato_nome_curto_retorna_422():
    prato_invalido = {
        "nome": "AB",
        "categoria": "pizza",
        "preco": 30.0
    }
    response = client.post("/pratos", json=prato_invalido)
    assert response.status_code == 422


def test_post_prato_categoria_invalida_retorna_422():
    prato_invalido = {
        "nome": "Prato Estranho",
        "categoria": "esoterico",
        "preco": 30.0
    }
    response = client.post("/pratos", json=prato_invalido)
    assert response.status_code == 422

## Exercício 3.5 — Desafio: cobertura de bebidas e pedidos 🎯

In [ ]:
# Arquivo: tests/test_pedidos.py

def test_criar_pedido_com_prato_existente(client):
    payload = {
        "prato_id": 1,
        "quantidade": 2,
        "observacao": "sem cebola"
    }
    response = client.post("/pedidos", json=payload)
    assert response.status_code in [200, 201]
    dados = response.json()
    assert "valor_total" in dados
    assert "nome_prato" in dados


def test_valor_total_calculado_corretamente(client):
    prato = client.get("/pratos/1").json()
    preco_unitario = prato["preco"]

    payload = {"prato_id": 1, "quantidade": 3}
    response = client.post("/pedidos", json=payload)
    assert response.status_code in [200, 201]
    assert response.json()["valor_total"] == preco_unitario * 3


def test_criar_pedido_com_prato_inexistente_retorna_404(client):
    payload = {"prato_id": 9999, "quantidade": 1}
    response = client.post("/pedidos", json=payload)
    assert response.status_code == 404


def test_criar_pedido_com_quantidade_zero_retorna_422(client):
    payload = {"prato_id": 1, "quantidade": 0}
    response = client.post("/pedidos", json=payload)
    assert response.status_code == 422

# BLOCO 4

## Exercício 4.1 — Criando o conftest.py

In [ ]:
import sys
from pathlib import Path

import pytest
from fastapi.testclient import TestClient

BASE_DIR = Path(__file__).resolve().parents[1]
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from main import app


@pytest.fixture
def client():
    """Cria um novo TestClient para os testes."""
    return TestClient(app)

In [ ]:
# Resposta:
# Centralizei a fixture client no conftest.py, evitando repetição de código
# e melhorando a organização da suíte de testes.

## Exercício 4.2 — Testes robustos vs. frágeis

In [ ]:
# Resposta:
# Testes frágeis assumem estado absoluto, como quantidade fixa de itens,
# posição exata em listas ou IDs específicos.
#
# Testes robustos verificam comportamento e estrutura, como:
# - status code
# - presença de campos esperados
# - coerência do filtro aplicado
# - resposta correta para entradas inválidas

## Exercício 4.3 — Parametrização de casos de erro

In [ ]:
# Trechos do arquivo tests/test_pratos.py após melhoria

import pytest


@pytest.mark.validacao
@pytest.mark.parametrize("categoria_invalida", [
    "esoterico",
    "fastfood",
    "japonesa",
    "PIZZA",
    "massa extra",
])
def test_categoria_invalida_retorna_422(client, categoria_invalida):
    prato = {
        "nome": "Prato Teste",
        "categoria": categoria_invalida,
        "preco": 40.0
    }
    response = client.post("/pratos", json=prato)
    assert response.status_code == 422


@pytest.mark.validacao
@pytest.mark.parametrize("preco_invalido", [-1.0, -0.01, -100.0])
def test_preco_invalido_retorna_422(client, preco_invalido):
    prato = {
        "nome": "Prato Teste",
        "categoria": "pizza",
        "preco": preco_invalido
    }
    response = client.post("/pratos", json=prato)
    assert response.status_code == 422


@pytest.mark.validacao
@pytest.mark.parametrize("id_inexistente", [9999, 123456, 99999])
def test_prato_inexistente_retorna_404(client, id_inexistente):
    response = client.get(f"/pratos/{id_inexistente}")
    assert response.status_code == 404

## Exercício 4.4 — Marcadores e pipeline seletivo

In [ ]:
# Arquivo: pytest.ini

[pytest]
markers =
    smoke: testes básicos que verificam se a API está no ar
    validacao: testes de validação de entrada
    integracao: testes que dependem de recursos externos

In [ ]:
# Resposta:
# Registrei os marcadores no pytest.ini, usando smoke para testes básicos
# e validacao para testes de entrada inválida.
# Isso ajuda a organizar melhor a suíte e a filtrar execuções no pipeline.

## Exercício 4.5 — Desafio: testando o contrato da API

In [ ]:
# Arquivo: tests/test_contratos.py
# A fixture 'client' vem do conftest.py — não é necessário redeclará-la aqui.


def test_contrato_get_prato(client):
    response = client.get("/pratos/1")
    assert response.status_code == 200
    prato = response.json()

    campos_obrigatorios = {"id", "nome", "categoria", "preco", "disponivel"}
    assert campos_obrigatorios.issubset(prato.keys())

    assert isinstance(prato["id"], int)
    assert isinstance(prato["nome"], str)
    assert isinstance(prato["categoria"], str)
    assert isinstance(prato["preco"], (int, float))
    assert isinstance(prato["disponivel"], bool)

    assert prato["preco"] > 0
    assert len(prato["nome"]) >= 3


def test_contrato_post_prato(client):
    novo = {
        "nome": "Prato Contrato Teste",
        "categoria": "massa",
        "preco": 45.0
    }

    response = client.post("/pratos", json=novo)
    assert response.status_code in [200, 201]
    prato = response.json()

    assert "id" in prato
    assert isinstance(prato["id"], int)
    assert prato["nome"] == "Prato Contrato Teste"
    assert prato["categoria"] == "massa"
    assert prato["preco"] == 45.0

    if "criado_em" in prato:
        assert isinstance(prato["criado_em"], str)
        assert len(prato["criado_em"]) > 0


def test_contrato_erro_404(client):
    response = client.get("/pratos/9999")
    assert response.status_code == 404
    corpo = response.json()

    assert "detail" in corpo or "erro" in corpo
    mensagem = corpo.get("detail") or corpo.get("erro")

    assert isinstance(mensagem, str)
    assert len(mensagem) > 0


def test_contrato_erro_422(client):
    response = client.post("/pratos", json={"nome": "X", "preco": -1})
    assert response.status_code == 422
    corpo = response.json()

    erros = corpo.get("detail") or corpo.get("detalhes")
    assert erros is not None
    assert isinstance(erros, list)
    assert len(erros) > 0

    if "detail" in corpo:
        for erro in erros:
            assert "loc" in erro
            assert "msg" in erro
            assert isinstance(erro["loc"], list)
            assert isinstance(erro["msg"], str)
            assert len(erro["msg"]) > 0